# 3. Ablation study

Trains the full model and each ablation variant with identical data and hyperparameters, then compares
them on the held-out split.

| Variant | What is removed |
|---|---|
| `wo_res` | Residual multi-scale convolutions (MFFM blocks) |
| `wo_dil` | Dilated gated convolution branch (WaveBlock) |
| `wo_conv` | The whole convolutional front end (1x1 projection straight into the transformer) |
| `wo_encoder` | Transformer encoder |
| `wo_decoder` | [CLS]-token transformer decoder (mean pooling instead) |
| `wo_trans` | Encoder and decoder (purely convolutional, global average pooling) |

In [ ]:
from pathlib import Path

# Run from the repository root or from notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASET = "bonn"   # "nmt", "bonn" or "tuev"
EPOCHS = None      # None = use the config value
DATA_ROOT = None   # None = REPO / <data_root from config>
VARIANTS = ["neurotransformer", "wo_res", "wo_dil", "wo_conv", "wo_encoder", "wo_decoder", "wo_trans"]
OUTPUT_DIR = REPO / "runs" / DATASET / "ablation"

In [ ]:
import pandas as pd
import torch

from neurotransformer.data import build_dataloaders, get_spec
from neurotransformer.engine import evaluate, fit
from neurotransformer.models import ABLATIONS, build_model
from neurotransformer.utils import (build_criterion, build_optimizer, build_scheduler, count_params,
                                    load_config, resolve_class_counts)

cfg = load_config(REPO / "configs" / f"{DATASET}.yaml")
if EPOCHS is not None:
    cfg["epochs"] = EPOCHS
data_root = Path(DATA_ROOT) if DATA_ROOT else REPO / cfg["data_root"]
spec = get_spec(DATASET)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaders = build_dataloaders(str(data_root), DATASET, cfg["batch_size"], cfg["num_workers"])
report_split = "test" if "test" in loaders else "eval"
cfg["loss"] = resolve_class_counts(cfg.get("loss"), loaders["train"].dataset)  # count once for all variants
print({split: len(loader.dataset) for split, loader in loaders.items()}, "-> reporting on", report_split)

In [ ]:
rows = []
for name in VARIANTS:
    print(f"\n=== {name}: {ABLATIONS.get(name, 'full model')} ===")
    model = build_model(name, spec.n_classes).to(device)
    criterion = build_criterion(cfg["loss"], device)
    optimizer = build_optimizer(model.parameters(), cfg["optimizer"])
    scheduler = build_scheduler(optimizer, cfg.get("scheduler"))
    checkpoint = OUTPUT_DIR / f"{name}.pth"

    fit(model, loaders["train"], loaders["eval"], optimizer, criterion, device, cfg["epochs"], spec.n_classes,
        str(checkpoint), scheduler, cfg.get("monitor", "eval_loss"), cfg.get("patience"), progress=False)

    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    metrics, _ = evaluate(model, loaders[report_split], device, spec.n_classes, criterion, progress=False)
    rows.append({"model": name, "removed": ABLATIONS.get(name, "-"), "params_m": count_params(model) / 1e6, **metrics})

results = pd.DataFrame(rows).set_index("model")
results.to_csv(OUTPUT_DIR / f"ablation_{report_split}.csv")
results[["removed", "params_m", "accuracy", "precision", "recall", "f1_score", "specificity", "auroc"]].round(4)

In [ ]:
import matplotlib.pyplot as plt

ax = results[["accuracy", "f1_score", "auroc"]].plot.bar(figsize=(10, 4), rot=30)
ax.set_ylim(max(0.0, results[["accuracy", "f1_score", "auroc"]].min().min() - 0.05), 1.0)
ax.set_title(f"Ablation on {DATASET} ({report_split} split)")
plt.tight_layout()